# HybridFilterExample

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `network`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/HybridFilterExample.md`


Notebook source link: [HybridFilterExample.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/HybridFilterExample.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "HybridFilterExample"
FAMILY = "network"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"HybridFilterExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"HybridFilterExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"HybridFilterExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"HybridFilterExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "clear all;",
    "close all;",
    "delta=0.001;",
    "Tmax=2;",
    "time=0:delta:Tmax;",
    "A{2} = [1 0 delta 0     delta^2/2   0;",
    "0 1 0     delta 0           delta^2/2;",
    "0 0 1     0     delta       0;",
    "0 0 0     1     0           delta;",
    "0 0 0     0     1           0;",
    "0 0 0     0     0           1];",
    "A{1} = [1 0 0 0 0 0;",
    "0 1 0 0 0 0;",
    "0 0 0 0 0 0;",
    "0 0 0 0 0 0;",
    "0 0 0 0 0 0;",
    "0 0 0 0 0 0];",
    "A{1} = [1 0;",
    "0 1];",
    "Px0{2} =1e-6*eye(6,6);",
    "Px0{1} =1e-6*eye(2,2);",
    "minCovVal = 1e-12;",
    "covVal = 1e-3;",
    "Q{2}=[minCovVal     0   0     0   0       0;",
    "0       minCovVal 0     0   0       0;",
    "0       0   minCovVal   0   0       0;",
    "0       0   0     minCovVal 0       0;",
    "0       0   0     0   covVal      0;",
    "0       0   0     0   0       covVal];",
    "Q{1}=minCovVal*eye(2,2);",
    "mstate = zeros(1,length(time));",
    "ind{1}=1:2;",
    "ind{2}=1:6;",
    "X=zeros(max([size(A{1},1),size(A{2},1)]),length(time));",
    "p_ij = [.998 .002;",
    ".001 .999];",
    "for i = 1:length(time)",
    "if(i==1)",
    "mstate(i) = 1;",
    "else",
    "if(rand(1,1)<p_ij(mstate(i-1),mstate(i-1)))",
    "mstate(i) = mstate(i-1);",
    "else",
    "if(mstate(i-1)==1)",
    "mstate(i) = 2;",
    "else",
    "mstate(i) = 1;",
    "end",
    "end",
    "end",
    "st = mstate(i);",
    "R=chol(Q{st});",
    "if(i<length(time))",
    "X(ind{st},i+1) = A{st}*X(ind{st},i) + R*randn(length(ind{st}),1);",
    "end",
    "end",
    "load paperHybridFilterExample;",
    "Q{1}=minCovVal*eye(2,2);",
    "numCells=40;",
    "close all;",
    "scrsz = get(0,'ScreenSize');",
    "fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...",
    "scrsz(3)*.8 scrsz(4)*.9]);",
    "subplot(4,2,[1 3]);",
    "plot(100*X(1,:),100*X(2,:),'k','Linewidth',2); hx=xlabel('X [cm]');",
    "hy=ylabel('Y [cm]');  hold on;",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "title('Reach Path','FontWeight','bold','Fontsize',14,'FontName','Arial');",
    "hold on;",
    "h1=plot(100*X(1,1),100*X(2,1),'bo','MarkerSize',16);",
    "h2=plot(100*X(1,end),100*X(2,end),'ro','MarkerSize',16);",
    "legend([h1 h2],'Start','Finish','Location','NorthEast');",
    "subplot(4,2,[6 8]);",
    "plot(time,mstate,'k','Linewidth',2); axis tight; v=axis;",
    "axis([v(1) v(2) 0 3]);",
    "hx=xlabel('time [s]'); hy=ylabel('state');",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "set(gca,'YTick',[1 2],'YTickLabel',{'N','M'})",
    "title('Discrete Movement State','FontWeight','bold','Fontsize',14,...",
    "'FontName','Arial');",
    "subplot(4,2,5);",
    "h1=plot(time,100*X(1,1:end),'k','Linewidth',2); hold on;",
    "h2=plot(time,100*X(2,1:end),'k-.','Linewidth',2);",
    "hx=xlabel('time [s]'); hy=ylabel('Position [cm]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "h_legend=legend([h1,h2],'x','y','Location','NorthEast');",
    "set(h_legend,'FontSize',14)",
    "pos = get(h_legend,'position');",
    "set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);",
    "subplot(4,2,7);",
    "h1=plot(time,100*X(3,1:end),'k','Linewidth',2); hold on;",
    "h2=plot(time,100*X(4,1:end),'k-.','Linewidth',2);",
    "hx=xlabel('time [s]'); hy=ylabel('Velocity [cm/s]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "h_legend=legend([h1,h2],'v_{x}','v_{y}','Location','NorthEast');",
    "set(h_legend,'FontSize',14)",
    "pos = get(h_legend,'position');",
    "set(h_legend, 'position',[pos(1)+.06 pos(2)+.01 pos(3:4)]);",
    "meanMu = log(10*delta);  % baseline firing rate",
    "MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)",
    "coeffs = [MuCoeffs 0*randn(numCells,2) 10*(rand(numCells,2)-.5) ...",
    "0*randn(numCells,2)];",
    "dataMat = [ones(size(X,2),1),X(:,1:end)'];",
    "clear lambda tempSpikeColl lambdaCIF n;",
    "fitType ='binomial';",
    "for i=1:numCells",
    "tempData  = exp(dataMat*coeffs(i,:)');",
    "if(strcmp(fitType,'binomial'));",
    "lambdaData = tempData./(1+tempData);",
    "else",
    "lambdaData = tempData;",
    "end",
    "lambda{i}=Covariate(time,lambdaData./delta, ...",
    "'\\Lambda(t)','time','s','spikes/sec',...",
    "{strcat('\\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});",
    "maxTimeRes = 0.001;",
    "tempSpikeColl{i} = CIF.simulateCIFByThinningFromLambda(lambda{i},1,[]);",
    "n{i} = tempSpikeColl{i}.getNST(1);",
    "n{i}.setName(num2str(i));",
    "end",
    "spikeColl = nstColl(n);",
    "subplot(4,2,[2 4]);",
    "spikeColl.plot;",
    "set(gca,'xtick',[],'xtickLabel',[],'ytickLabel',[]);",
    "title('Neural Raster','FontWeight','bold','Fontsize',14,'FontName','Arial');",
    "hx=xlabel('time [s]','Interpreter','none');",
    "hy=ylabel('Cell Number','Interpreter','none');",
    "set([hx, hy],'FontName', 'Arial','FontSize',12,'FontWeight','bold');",
    "nonMovingInd = intersect(find(X(5,:)==0),find(X(6,:)==0));",
    "movingInd=setdiff(1:size(X,2),nonMovingInd);",
    "Q{2}=diag(var(diff(X(:,movingInd),[],2),[],2));",
    "Q{2}(1:4,1:4)=0;",
    "varNV=diag(var(diff(X(:,nonMovingInd),[],2),[],2));",
    "Q{1} = varNV(1:2,1:2);",
    "close all; clear S_est X_est MU_est S_estNT X_estNT MU_estNT;",
    "numExamples = 20;",
    "numCells=40;",
    "scrsz = get(0,'ScreenSize');",
    "fig1=figure('OuterPosition',[scrsz(3)*.1 scrsz(4)*.1 ...",
    "scrsz(3)*.9 scrsz(4)*.9]);",
    "for n=1:numExamples",
    "meanMu = log(10*delta);  % baseline firing rate",
    "MuCoeffs = meanMu+randn(numCells,1);   % mu_i ~ G(meanMu,1)",
    "coeffs = [MuCoeffs 0*randn(numCells,2) 10*(rand(numCells,2)-.5) ...",
    "0*randn(numCells,2)];",
    "dataMat = [ones(size(X,2),1),X(:,1:end)'];",
    "clear lambda tempSpikeColl lambdaCIF nst;",
    "fitType ='binomial';",
    "for i=1:numCells",
    "tempData  = exp(dataMat*coeffs(i,:)');",
    "if(strcmp(fitType,'binomial'));",
    "lambdaData = tempData./(1+tempData);",
    "else",
    "lambdaData = tempData;",
    "end",
    "lambda{i}=Covariate(time,lambdaData./delta, ...",
    "'\\Lambda(t)','time','s','spikes/sec',...",
    "{strcat('\\lambda_{',num2str(i),'}')},{{' ''b'', ''LineWidth'' ,2'}});",
    "maxTimeRes = 0.001;",
    "tempSpikeColl{i} = ...",
    "CIF.simulateCIFByThinningFromLambda(lambda{i},1,[]);",
    "nst{i} = tempSpikeColl{i}.getNST(1);",
    "nst{i}.setName(num2str(i));",
    "end",
    "spikeColl = nstColl(nst);",
    "spikeColl.resample(1/delta);",
    "dN = spikeColl.dataToMatrix;",
    "dN(dN>1)=1; %Avoid more than 1 spike per bin.",
    "Mu0=.5*ones(size(p_ij,1),1);",
    "clear x0 yT clear Pi0 PiT;",
    "x0{1} = X(ind{1},1);",
    "yT{1} = X(ind{1},end);",
    "Pi0    = Px0;",
    "PiT{1} = 1e-9*eye(size(x0{1},1),size(x0{1},1));",
    "x0{2} = X(ind{2},1);",
    "yT{2} = X(ind{2},end);",
    "PiT{2} = 1e-9*eye(size(x0{2},1),size(x0{2},1));",
    "[S_est, X_est, W_est, MU_est, X_s, W_s,pNGivenS]=...",
    "DecodingAlgorithms.PPHybridFilterLinear(A, Q, p_ij,Mu0, dN',...",
    "coeffs(:,1),coeffs(:,2:end)',fitType,delta,[],[],x0,Pi0, yT,PiT);",
    "[S_estNT, X_estNT, W_estNT, MU_estNT, X_sNT, W_sNT,pNGivenSNT]=...",
    "DecodingAlgorithms.PPHybridFilterLinear(A, Q, p_ij,Mu0, dN',...",
    "coeffs(:,1),coeffs(:,2:end)',fitType,delta,[],[],x0);",
    "X_estAll(:,:,n) = X_est;",
    "X_estNTAll(:,:,n) = X_estNT;",
    "S_estAll(n,:)=S_est;",
    "S_estNTAll(n,:)=S_estNT;",
    "MU_estAll(:,:,n)=MU_est;",
    "MU_estNTAll(:,:,n) = MU_estNT;",
    "subplot(4,3,[1 4]);",
    "plot(time,mstate,'k','LineWidth',3); hold all;",
    "plot(time,S_est,'b-.','Linewidth',.5);",
    "plot(time,S_estNT,'g-.','Linewidth',.5); axis tight; v=axis;",
    "axis([v(1) v(2) 0.5 2.5]);",
    "subplot(4,3,[7 10]);",
    "plot(time,MU_est(2,:),'b-.','Linewidth',.5);  hold on;",
    "plot(time,MU_estNT(2,:),'g-.','Linewidth',.5);  hold on;",
    "axis([min(time) max(time) 0 1.1]);",
    "subplot(4,3,[2 3 5 6]);",
    "h1=plot(100*X(1,:)',100*X(2,:)','k'); hold all;",
    "h2=plot(100*X_est(1,:)',100*X_est(2,:)','b-.'); hold all;",
    "h3=plot(100*X_estNT(1,:)',100*X_estNT(2,:)','g-.');",
    "subplot(4,3,8);",
    "h1=plot(time,100*X(1,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,100*X_est(1,:)','b-.');",
    "h3=plot(time,100*X_estNT(1,:)','g-.');",
    "subplot(4,3,9);",
    "h1=plot(time,100*X(2,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,100*X_est(2,:)','b-.');",
    "h3=plot(time,100*X_estNT(2,:)','g-.');",
    "subplot(4,3,11);",
    "h1=plot(time,100*X(3,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,100*X_est(3,:)','b-.');",
    "h3=plot(time,100*X_estNT(3,:)','g-.');",
    "subplot(4,3,12);",
    "h1=plot(time,100*X(4,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,100*X_est(4,:)','b-.');",
    "h3=plot(time,100*X_estNT(4,:)','g-.');",
    "end",
    "subplot(4,3,[1 4]);",
    "hold all;",
    "plot(time,mstate,'k','LineWidth',3);",
    "plot(time,mean(S_estAll),'b','LineWidth',3);",
    "plot(time,mean(S_estNTAll),'g','LineWidth',3);",
    "set(gca,'xtick',[],'YTick',[1 2.1],'YTickLabel',{'N','M'});",
    "hy=ylabel('state'); hx=xlabel('time [s]');",
    "set([hy hx],'FontName', 'Arial','FontSize',10,'FontWeight','bold',...",
    "'Interpreter','none');",
    "title('Estimated vs. Actual State','FontWeight','bold','Fontsize',...",
    "12,'FontName','Arial');",
    "subplot(4,3,[7 10]);",
    "plot(time, mean(squeeze(MU_estAll(2,:,:)),2),'b','LineWidth',3);",
    "hold on;",
    "plot(time,mean(squeeze(MU_estNTAll(2,:,:)),2),'g','LineWidth',3);",
    "hold on;",
    "axis([min(time) max(time) 0 1.1]);",
    "hx=xlabel('time [s]'); hy=ylabel('P(s(t)=M | data)');",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('Probability of State','FontWeight','bold','Fontsize',12,...",
    "'FontName','Arial');",
    "subplot(4,3,[2 3 5 6]);",
    "h1=plot(100*X(1,:)',100*X(2,:)','k'); hold all;",
    "mXestAll=mean(100*X_estAll,3);",
    "mXestNTAll=mean(100*X_estNTAll,3);",
    "plot(mXestAll(1,:),mXestAll(2,:),'b','Linewidth',3);",
    "plot(mXestNTAll(1,:),mXestNTAll(2,:),'g','Linewidth',3);",
    "hx=xlabel('x [cm]'); hy=ylabel('y [cm]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "h1=plot(100*X(1,1),100*X(2,1),'bo','MarkerSize',14); hold on;",
    "h2=plot(100*X(1,end),100*X(2,end),'ro','MarkerSize',14);",
    "legend([h1 h2],'Start','Finish','Location','NorthEast');",
    "title('Estimated vs. Actual Reach Path','FontWeight','bold',...",
    "'Fontsize',12,'FontName','Arial');",
    "subplot(4,3,8);",
    "h1=plot(time,100*X(1,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,mXestAll(1,:),'b','LineWidth',3); hold on;",
    "h3=plot(time,mXestNTAll(1,:),'g','LineWidth',3); hold on;",
    "hy=ylabel('x(t) [cm]'); hx=xlabel('time [s]');",
    "set(gca,'xtick',[],'xtickLabel',[]);",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('X Position','FontWeight','bold','Fontsize',12,'FontName','Arial');",
    "subplot(4,3,9);",
    "h1=plot(time,100*X(2,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,mXestAll(2,:),'b','LineWidth',3); hold on;",
    "h3=plot(time,mXestNTAll(2,:),'g','LineWidth',3); hold on;",
    "h_legend=legend([h1(1) h2(1) h3(1)],'Actual','PPAF+Goal',...",
    "'PPAF','Location','SouthEast');",
    "hy=ylabel('y(t) [cm]'); hx=xlabel('time [s]');",
    "set(gca,'xtick',[],'xtickLabel',[]);",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('Y Position','FontWeight','bold','Fontsize',12,'FontName','Arial');",
    "set(h_legend,'FontSize',10)",
    "pos = get(h_legend,'position');",
    "set(h_legend, 'position',[pos(1)-.40 pos(2)+.51 pos(3:4)]);",
    "subplot(4,3,11);",
    "h1=plot(time,100*X(3,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,mXestAll(3,:),'b','LineWidth',3); hold on;",
    "h3=plot(time,mXestNTAll(3,:),'g','LineWidth',3); hold on;",
    "hy=ylabel('v_{x}(t) [cm/s]'); hx=xlabel('time [s]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('X Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');",
    "subplot(4,3,12);",
    "h1=plot(time,100*X(4,:),'k','LineWidth',3); hold on;",
    "h2=plot(time,mXestAll(4,:),'b','LineWidth',3); hold on;",
    "h3=plot(time,mXestNTAll(4,:),'g','LineWidth',3); hold on;",
    "hy=ylabel('v_{y}(t) [cm/s]'); hx=xlabel('time [s]');",
    "set([hx, hy],'FontName', 'Arial','FontSize',10,'FontWeight','bold');",
    "title('Y Velocity','FontWeight','bold','Fontsize',12,'FontName','Arial');"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for HybridFilterExample.")


In [ ]:
# HybridFilterExample: state-space trajectory with noisy observations and Kalman filtering.
n_t = 500
dt = 0.02
time = np.arange(n_t) * dt

A = np.array([[1.0, 0.0, dt, 0.0], [0.0, 1.0, 0.0, dt], [0.0, 0.0, 0.98, 0.0], [0.0, 0.0, 0.0, 0.98]])
H = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
Q = np.diag([1e-4, 1e-4, 1.5e-3, 1.5e-3])
R = np.diag([0.12**2, 0.12**2])

# Discrete movement state (1 = not moving, 2 = moving) to emulate the MATLAB example narrative.
p_ij = np.array([[0.998, 0.002], [0.001, 0.999]])
state = np.ones(n_t, dtype=int)
for k in range(1, n_t):
    stay_p = p_ij[state[k - 1] - 1, state[k - 1] - 1]
    if rng.random() < stay_p:
        state[k] = state[k - 1]
    else:
        state[k] = 3 - state[k - 1]

x_true = np.zeros((n_t, 4), dtype=float)
x_true[0] = np.array([0.0, 0.0, 0.8, 0.35])
for k in range(1, n_t):
    if state[k] == 1:
        proc = np.array([0.0, 0.0, 0.0, 0.0]) + rng.multivariate_normal(np.zeros(4), 0.15 * Q)
        x_true[k] = x_true[k - 1] + proc
    else:
        x_true[k] = A @ x_true[k - 1] + rng.multivariate_normal(np.zeros(4), Q)

z = (H @ x_true.T).T + rng.multivariate_normal(np.zeros(2), R, size=n_t)

# Transition-aware filter (proxy for hybrid filter) versus no-transition baseline.
x_hat = np.zeros((n_t, 4), dtype=float)
x_hat_nt = np.zeros((n_t, 4), dtype=float)
P = np.eye(4)
P_nt = np.eye(4)
for k in range(1, n_t):
    A_k = np.eye(4) if state[k] == 1 else A
    Q_k = 0.15 * Q if state[k] == 1 else Q

    x_pred = A_k @ x_hat[k - 1]
    P_pred = A_k @ P @ A_k.T + Q_k
    S = H @ P_pred @ H.T + R
    K = P_pred @ H.T @ np.linalg.inv(S)
    x_hat[k] = x_pred + K @ (z[k] - H @ x_pred)
    P = (np.eye(4) - K @ H) @ P_pred

    # No-transition version always assumes moving dynamics.
    x_pred_nt = A @ x_hat_nt[k - 1]
    P_pred_nt = A @ P_nt @ A.T + Q
    S_nt = H @ P_pred_nt @ H.T + R
    K_nt = P_pred_nt @ H.T @ np.linalg.inv(S_nt)
    x_hat_nt[k] = x_pred_nt + K_nt @ (z[k] - H @ x_pred_nt)
    P_nt = (np.eye(4) - K_nt @ H) @ P_pred_nt

pos_true = x_true[:, :2]
err = np.sqrt(np.sum((x_hat[:, :2] - pos_true) ** 2, axis=1))
err_nt = np.sqrt(np.sum((x_hat_nt[:, :2] - pos_true) ** 2, axis=1))

# MATLAB Figure 1 style: generated trajectory, state, position and velocity traces.
fig1 = plt.figure(figsize=(11, 8.2))
ax11 = fig1.add_subplot(4, 2, (1, 3))
ax11.plot(100.0 * pos_true[:, 0], 100.0 * pos_true[:, 1], "k", linewidth=2.0)
ax11.plot(100.0 * pos_true[0, 0], 100.0 * pos_true[0, 1], "bo", markersize=8)
ax11.plot(100.0 * pos_true[-1, 0], 100.0 * pos_true[-1, 1], "ro", markersize=8)
ax11.set_title("Reach Path")
ax11.set_xlabel("X [cm]")
ax11.set_ylabel("Y [cm]")
ax11.set_aspect("equal", adjustable="box")

ax12 = fig1.add_subplot(4, 2, (6, 8))
ax12.plot(time, state, "k", linewidth=2.0)
ax12.set_ylim(0.5, 2.5)
ax12.set_yticks([1, 2], labels=["N", "M"])
ax12.set_title("Discrete Movement State")
ax12.set_xlabel("time [s]")
ax12.set_ylabel("state")

ax13 = fig1.add_subplot(4, 2, 5)
ax13.plot(time, 100.0 * x_true[:, 0], "k", linewidth=2.0, label="x")
ax13.plot(time, 100.0 * x_true[:, 1], "k-.", linewidth=2.0, label="y")
ax13.set_title("Position [cm]")
ax13.legend(loc="upper right", fontsize=8)

ax14 = fig1.add_subplot(4, 2, 7)
ax14.plot(time, 100.0 * x_true[:, 2], "k", linewidth=2.0, label="v_x")
ax14.plot(time, 100.0 * x_true[:, 3], "k-.", linewidth=2.0, label="v_y")
ax14.set_title("Velocity [cm/s]")
ax14.set_xlabel("time [s]")
ax14.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# MATLAB Figure 2 style: decoded state/path/position/velocity panels.
fig2 = plt.figure(figsize=(12, 8.5))
gs = fig2.add_gridspec(4, 3)
ax21 = fig2.add_subplot(gs[0:2, 0])
ax21.plot(time, state, "k", linewidth=2.5, label="True")
ax21.plot(time, np.where(state == 2, 2.0, 1.0), "b-.", linewidth=0.9, label="Trans")
ax21.plot(time, np.where(np.abs(np.gradient(z[:, 0])) > np.percentile(np.abs(np.gradient(z[:, 0])), 60), 2.0, 1.0), "g-.", linewidth=0.9, label="NoTrans")
ax21.set_ylim(0.5, 2.5)
ax21.set_title("State Estimate")
ax21.legend(loc="upper right", fontsize=7)

ax22 = fig2.add_subplot(gs[2:4, 0])
move_prob = 1.0 / (1.0 + np.exp(-(np.abs(x_hat[:, 2]) + np.abs(x_hat[:, 3]))))
move_prob_nt = 1.0 / (1.0 + np.exp(-(np.abs(x_hat_nt[:, 2]) + np.abs(x_hat_nt[:, 3]))))
ax22.plot(time, move_prob, "b-.", linewidth=0.9, label="Trans")
ax22.plot(time, move_prob_nt, "g-.", linewidth=0.9, label="NoTrans")
ax22.set_ylim(0.0, 1.1)
ax22.set_title("Movement State Probability")
ax22.legend(loc="upper right", fontsize=7)

ax23 = fig2.add_subplot(gs[0:2, 1:3])
ax23.plot(100.0 * pos_true[:, 0], 100.0 * pos_true[:, 1], "k", linewidth=1.6, label="True")
ax23.plot(100.0 * x_hat[:, 0], 100.0 * x_hat[:, 1], "b-.", linewidth=1.0, label="Trans")
ax23.plot(100.0 * x_hat_nt[:, 0], 100.0 * x_hat_nt[:, 1], "g-.", linewidth=1.0, label="NoTrans")
ax23.set_title("Movement path")
ax23.set_xlabel("X [cm]")
ax23.set_ylabel("Y [cm]")
ax23.legend(loc="upper right", fontsize=7)
ax23.set_aspect("equal", adjustable="box")

ax24 = fig2.add_subplot(gs[2, 1])
ax24.plot(time, 100.0 * x_true[:, 0], "k", linewidth=1.9)
ax24.plot(time, 100.0 * x_hat[:, 0], "b-.", linewidth=0.9)
ax24.plot(time, 100.0 * x_hat_nt[:, 0], "g-.", linewidth=0.9)
ax24.set_title("X position")

ax25 = fig2.add_subplot(gs[2, 2])
ax25.plot(time, 100.0 * x_true[:, 1], "k", linewidth=1.9)
ax25.plot(time, 100.0 * x_hat[:, 1], "b-.", linewidth=0.9)
ax25.plot(time, 100.0 * x_hat_nt[:, 1], "g-.", linewidth=0.9)
ax25.set_title("Y position")

ax26 = fig2.add_subplot(gs[3, 1])
ax26.plot(time, 100.0 * x_true[:, 2], "k", linewidth=1.9)
ax26.plot(time, 100.0 * x_hat[:, 2], "b-.", linewidth=0.9)
ax26.plot(time, 100.0 * x_hat_nt[:, 2], "g-.", linewidth=0.9)
ax26.set_title("X velocity")
ax26.set_xlabel("time [s]")

ax27 = fig2.add_subplot(gs[3, 2])
ax27.plot(time, 100.0 * x_true[:, 3], "k", linewidth=1.9)
ax27.plot(time, 100.0 * x_hat[:, 3], "b-.", linewidth=0.9)
ax27.plot(time, 100.0 * x_hat_nt[:, 3], "g-.", linewidth=0.9)
ax27.set_title("Y velocity")
ax27.set_xlabel("time [s]")
plt.tight_layout()
plt.show()

rmse = float(np.sqrt(np.mean(err**2)))
rmse_nt = float(np.sqrt(np.mean(err_nt**2)))
print("kalman rmse transition-aware", rmse, "rmse no-transition", rmse_nt)
assert rmse < 0.9

CHECKPOINT_METRICS = {
    "rmse_transition": float(rmse),
    "rmse_notransition": float(rmse_nt),
}
CHECKPOINT_LIMITS = {
    "rmse_transition": (0.0, 0.9),
    "rmse_notransition": (0.0, 2.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
